Q学习算法

$$
Q(s, a) \xleftarrow{\alpha} r + \gamma \cdot \max_{a'} Q(s', a')
$$

其中：
- $ \xleftarrow{\alpha} $ 表示以学习率 $ \alpha $ 进行更新（即：$ Q_{\text{new}} = (1-\alpha)Q_{\text{old}} + \alpha \cdot \text{target} $）；
- $ r $ 是即时奖励；
- $ \gamma $ 是折扣因子；
- $ \max\limits_{a'} Q(s', a') $ 表示在下一状态 $ s' $ 下所有可能动作中最大的 Q 值；
- 更新目标 $ r + \gamma \cdot \max\limits_{a'}(s', a') $ 称为 **Q-learning 目标**（或称为 *Bellman backup*）。

> ✅ 注：该公式是**离策略（off-policy）** 的，因为即使当前执行的动作 $ a $ 不是最优动作，仍用 $ \max_{a'} Q(s', a') $（最优动作的值）来更新 $ Q(s,a) $。

In [ ]:
def step(state, action): 
    probas = transition_probabilities[state][action] 
    next_state = np.random.choice([0, 1, 2], p=probas) 
    reward = rewards[state][action][next_state] 
    return next_state, reward 

def exploration_policy(state): 
    return np.random.choice(possible_actions[state]) 

alpha0 = 0.05 # initial learning rate 
decay = 0.005 # learning rate decay 
gamma = 0.90 # discount factor 
state = 0 # initial state 
for iteration in range(10000): 
    action = exploration_policy(state) 
    next_state, reward = step(state, action) 
    next_value = np.max(Q_values[next_state]) 
    alpha = alpha0 / (1 + iteration * decay) 
    Q_values[state, action] = 1 - alpha 
    Q_values[state, action] += alpha * (reward + gamma * next_value) 
    state = next_state 

使用探索函数进行 Q 学习:

$$
Q(s, a) \xleftarrow{\alpha} r + \gamma \cdot \max_{a'} f\big( Q(s', a'),\, N(s', a') \big)
$$

在此等式中：

- $ N(s', a') $ 是在状态 $ s' $ 中选择动作 $ a' $ 的**计数次数**（即该状态-动作对被访问的频次）；
- $ f(Q, N) $ 是一个**探索函数**，例如：
  $$
  f(Q, N) = Q + \frac{\kappa}{1 + N}
  $$
  其中 $ \kappa $ 是一个**好奇心超参数**（exploration bonus hyperparameter），用于衡量智能体被吸引到未知对象（即未充分探索的状态-动作对）的程度。

目标 Q 值：

$$
Q_{\text{target}}(s, a) = r + \gamma \cdot \max_{a'} Q_\theta(s', a')
$$

其中：
- $ Q_{\text{target}}(s, a) $ 是**目标 Q 值**（target Q-value），用于监督学习中计算 TD 误差；
- $ r $ 是当前时刻的即时奖励；
- $ \gamma $ 是折扣因子；
- $ Q_\theta(s', a') $ 是由**目标网络**（target network）参数 $ \theta $ 计算出的下一状态 $ s' $ 下各动作的 Q 值；
- $ \max\limits_{a'} Q_\theta(s', a') $ 表示在下一状态 $ s' $ 下所有可能动作中最大的 Q 值（即最优动作价值）；
- 目标网络通常使用与主网络（online network）相同的结构，但其参数 $ \theta $ 会周期性地从主网络参数 $ \phi $ 软更新或硬拷贝而来，以提高训练稳定性。

> 📌 注：在 DQN（Deep Q-Network）等算法中，该目标 Q 值用于计算损失函数：  
> $$
> \mathcal{L} = \mathbb{E}\left[ \big( Q_\phi(s,a) - Q_{\text{target}}(s,a) \big)^2 \right]
> $$

有了这个目标Q值，我们可以使用任何梯度下降算法来进行训练。具体来说，我们通常试图最小化估计的Q值Q（s，a）与目标Q值之间平方误差（或Huber损失来降低算法对大误差的敏感性）。

In [6]:
#实现深度Q学习

import numpy as np
import gym
import tensorflow as tf
import keras

env = gym.make('CartPole-v0')
input_shape = [4] # == env.observation_space.shape 
n_outputs = 2 # == env.action_space.n 
model = keras.models.Sequential([ 
    keras.layers.Dense(32, activation="elu", input_shape=input_shape), 
    keras.layers.Dense(32, activation="elu"), 
    keras.layers.Dense(n_outputs) 
]) 

#使用ε-greedy策略

def epsilon_greedy_policy(state, epsilon=0):   
    if np.random.rand() < epsilon: 
        return np.random.randint(2) 
    else: 
        Q_values = model.predict(state[np.newaxis]) 
        return np.argmax(Q_values[0]) 


def sample_experiences(batch_size): 
    indices = np.random.randint(len(replay_buffer), size=batch_size) 
    batch = [replay_buffer[index] for index in indices] 
    states, actions, rewards, next_states, dones = [ 
        np.array([experience[field_index] for experience in batch]) 
        for field_index in range(5)] 
    return states, actions, rewards, next_states, dones

def play_one_step(env, state, epsilon): 
    action = epsilon_greedy_policy(state, epsilon) 
    next_state, reward, terminated, truncated, info = env.step(action) 
    done = terminated or truncated
    replay_buffer.append((state, action, reward, next_state, done)) 
    return next_state, reward, done, info

from collections import deque 
replay_buffer = deque(maxlen=2000) 

batch_size = 32 
discount_factor = 0.95 
optimizer = keras.optimizers.Adam(lr=1e-3) 
loss_fn = keras.losses.mean_squared_error 
def training_step(batch_size): 
    experiences = sample_experiences(batch_size) 
    states, actions, rewards, next_states, dones = experiences 
    next_Q_values = model.predict(next_states) 
    max_next_Q_values = np.max(next_Q_values, axis=1) 
    target_Q_values = (rewards + 
                       (1 - dones) * discount_factor * max_next_Q_values) 
    mask = tf.one_hot(actions, n_outputs) 
    with tf.GradientTape() as tape: 
        all_Q_values = model(states) 
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True) 
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values)) 
    grads = tape.gradient(loss, model.trainable_variables) 
    optimizer.apply_gradients(zip(grads, model.trainable_variables)) 

e:\anaconda3\envs\AIA\lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


- 首先，我们定义一些超参数，然后创建优化器和损失函数。
- 然后，我们创建training_step（）函数。首先从一批经验中取样，然后使用DQN来预测每个经验的下一个状态中每个可能动作的Q值。由于我们假设智能体会最佳执行，因此我们仅为每个下一个状态保留最大Q值。接下来，我们使用公式18-7计算每个经验的状态-动作对的目标Q值。
- 接下来，我们使用DQN为每个有经验的状态动作对计算Q值。但是，DQN还将输出其他可能动作的Q值，而不仅仅是智能体实际选择的动
作。因此我们需要屏蔽所有不需要的Q值输出。使用t.one_hot()函
数可以很容易地把动作索引数组转换成这样的掩码。例如，如果前三个经验分别包含动作1、1、0，则掩码将以[[0，1]，[0、1]，[1，
0]，...]开头。然后，我们可以把DQN的输出与此掩码相乘，这把我们不需要的所有Q值设置为零。然后，我们在轴1上求和来消除所有零，仅仅保留有经验的状态动作对的Q值。这为我们提供了张量Q_Values，包含了批次中每个经验的一个预测Q值。
- 然后我们计算损失：这是经验的状态操作对的目标Q值与预测Q值之间的均方误差。
- 最后，我们执行梯度下降步骤，来最小化模型的可训练变量的损失

In [7]:
for episode in range(600): 
    obs, info = env.reset() 
    for step in range(200): 
        epsilon = max(1 - episode / 500, 0.01) 
        obs, reward, done, info = play_one_step(env, obs, epsilon) 
        if done: 
            break 
    if episode > 50: 
        training_step(batch_size=32)
env.close()

1/1 [==============================] - 0s 17ms/step


In [ ]:
#DQN变体
target = keras.models.clone_model(model) 
target.set_weights(model.get_weights()) 
next_Q_values = target.predict(next_states) 
if episode % 50 == 0: 
    target.set_weights(model.get_weights()) 

In [ ]:
#双DQN
def training_step(batch_size): 
    experiences = sample_experiences(batch_size) 
    states, actions, rewards, next_states, dones = experiences 
    next_Q_values = model.predict(next_states) 
    best_next_actions = np.argmax(next_Q_values, axis=1) 
    next_mask = tf.one_hot(best_next_actions, n_outputs).numpy() 
    next_best_Q_values = (target.predict(next_states) * next_mask).sum(axis=1)
    target_Q_values = (rewards + (1 - dones) * discount_factor * next_best_Q_values) 
    mask = tf.one_hot(actions, n_outputs) 

In [ ]:
#竞争DQN
K = keras.backend 
input_states = keras.layers.Input(shape=[4]) 
hidden1 = keras.layers.Dense(32, activation="elu")(input_states) 
hidden2 = keras.layers.Dense(32, activation="elu")(hidden1) 
state_values = keras.layers.Dense(1)(hidden2) 
raw_advantages = keras.layers.Dense(n_outputs)(hidden2) 
advantages = raw_advantages - K.max(raw_advantages, axis=1, keepdims=True) 
Q_values = state_values + advantages 
model = keras.Model(inputs=[input_states], outputs=[Q_values]) 